# 07 — ANNOVAR output, gnomAD, consequence, and SpliceAI filtering

This notebook processes each run's external ANNOVAR multianno output.

For each selected run, it:

- filters variants using gnomAD v4.1 exome and genome allele frequencies;
- retains the existing loss-of-function and splice consequence categories;
- applies the SpliceAI threshold only to pure-splicing variants;
- saves intermediate, final, removed-variant, and filtering-summary outputs.

All original ANNOVAR and per-variant columns are retained. Variants are not deduplicated.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

notebook_dir = Path("/home/donetski/Notebooks")
annovar_dir = notebook_dir / "OutputFiles" / "06_annovar_output"
notebook04_dir = notebook_dir / "OutputFiles" / "04_qc_checking_on_target"
#output_dir = notebook_dir / "OutputFiles" / "07_gnomad_consequence_spliceai"
output_dir = Path("/project/knathans_shared/donetski/Notebooks/OutputFiles/FinalPVs")

output_dir.mkdir(parents=True, exist_ok=True)
nmd_boundary_csv = Path(r"/home/donetski/Notebooks/InputFiles/complete_clean_penultimate_exon_annotation_v2025.csv")

target_output_dir = output_dir / "16_target_genes"
target_output_dir.mkdir(parents=True, exist_ok=True)

## Select runs and filtering settings

Set `runs_to_process` to one run, several runs, or `"all"`.

The gnomAD threshold is 0.005, corresponding to 0.5%. A value equal to 0.005 does not pass.

In [ ]:
output_prefix = "07"
runs_to_process = "Run2"  # "all", "Run2", or ["Run2", "Run4"]

all_runs = ["Run1", "Run2", "Run3", "Run4"]
runs = all_runs if isinstance(runs_to_process, str) and runs_to_process.lower() == "all" else [runs_to_process] if isinstance(runs_to_process, str) else runs_to_process

gnomad_threshold = 0.005
spliceai_threshold = 0.5

spliceai_cols = ["SpliceAI.DS_AG", "SpliceAI.DS_AL", "SpliceAI.DS_DG", "SpliceAI.DS_DL"]
TARGET_GENES = ["ATM", "BARD1", "BRCA1", "BRCA2", "CDH1", "CDKN2A", "CHEK2", "MLH1", "MSH2", "MSH6", "PALB2", "PMS2", "PTEN", "RAD51C", "RAD51D", "TP53"]

lof_terms = [
    "stop_gained",
    "frameshift_variant",
    "splice_acceptor_variant",
    "splice_donor_variant",
    "start_lost",
]

severe_terms = {
    "frameshift_variant",
    "stop_gained",
    "start_lost",
    "stop_lost",
    "protein_altering_variant",
    "inframe_insertion",
    "inframe_deletion",
}

print("Runs to process:", runs)
print("gnomAD threshold:", gnomad_threshold)
print("SpliceAI threshold:", spliceai_threshold)

## Add SpliceAI scores

The SpliceAI filter needs the four delta score columns: `SpliceAI_DS_AG`, `SpliceAI_DS_AL`, `SpliceAI_DS_DG`, and `SpliceAI_DS_DL`.

If those columns are already present in the ANNOVAR dataframe, they are used directly. If they are missing, the notebook loads the earlier per-variant file for the same run and merges the SpliceAI scores onto the ANNOVAR-filtered rows using chromosome, position, reference allele, and alternate allele.

In [ ]:
def get_run_files(run):
    #annovar_file = annovar_dir / f"fixed_annovar_output.csv"
    original_file = notebook04_dir / f"04_{run}_per_variant_target_status_FULL.csv"
    input_file = Path("/project/knathans_shared/donetski/Notebooks/OutputFiles/FinalPVs/unique_variants_target_genes_sample_count_lt50_VAF_ge1pct_AltDepth_ge5.csv")
    #return annovar_file, original_file
    return input_file, original_file

## gnomAD filtering

All columns beginning with `gnomad41_exome_AF` or `gnomad41_genome_AF` are evaluated.

Unavailable values such as `.`, blanks, `NA`, and `N/A` do not remove a row. Every available numeric allele frequency must be strictly less than 0.005.

In [ ]:
def filter_gnomad(df):
    exome_cols = [col for col in df.columns if col.startswith("gnomad41_exome_AF")]
    genome_cols = [col for col in df.columns if col.startswith("gnomad41_genome_AF")]
    gnomad_cols = exome_cols + genome_cols

    if not exome_cols:
        raise ValueError("No gnomad41_exome_AF columns were found.")
    if not genome_cols:
        raise ValueError("No gnomad41_genome_AF columns were found.")

    numeric_af = df[gnomad_cols].replace([".", "", " ", "N/A", "NA", "nan", "NaN"], pd.NA).apply(pd.to_numeric, errors="coerce")
    passes_gnomad = (numeric_af.isna() | numeric_af.lt(gnomad_threshold)).all(axis=1)

    return df.loc[passes_gnomad].copy(), exome_cols, genome_cols

## Consequence filtering

Retain variants whose `Variant.Consequence` contains at least one of the original LoF or splice terms.

This also detects terms inside compound annotations such as:

```text
splice_donor_variant&intron_variant

In [ ]:
def filter_consequences(df):
    pattern = "|".join(lof_terms)
    passes_consequence = df["Variant.Consequence"].fillna("").str.contains(pattern, regex=True)
    return df.loc[passes_consequence].copy()

## Restore SpliceAI scores only when necessary

If all four SpliceAI score columns are already present in the ANNOVAR output, they are used directly.

Otherwise, the scores are restored from the notebook-04 table using the original anchored variant fields:

```text
Chr, Start, REF, ALT

In [ ]:
def normalize_match_columns(df, chr_col, start_col, ref_col, alt_col):
    keys = df[[chr_col, start_col, ref_col, alt_col]].copy()
    keys.columns = ["match_chr", "match_start", "match_ref", "match_alt"]

    keys["match_chr"] = keys["match_chr"].astype("string").str.strip().str.replace(r"^chr", "", regex=True, case=False)
    keys["match_start"] = pd.to_numeric(keys["match_start"], errors="coerce").astype("Int64")
    keys["match_ref"] = keys["match_ref"].astype("string").str.strip().str.upper()
    keys["match_alt"] = keys["match_alt"].astype("string").str.strip().str.upper()

    return keys

In [ ]:
#checks which naming scheme the DataFrame uses
def find_original_variant_columns(df):
    if all(col in df.columns for col in ["Chr.1", "Start.1", "REF", "ALT"]):
        return "Chr.1", "Start.1", "REF", "ALT"

    if all(col in df.columns for col in ["Chr", "Start", "REF", "ALT"]):
        return "Chr", "Start", "REF", "ALT"

    raise ValueError("Could not identify the original anchored Chr, Start, REF, and ALT columns.")

In [ ]:
def clean_spliceai_scores(df):
    result = df.copy()
    for col in spliceai_cols:
        result[col] = pd.to_numeric(result[col].replace(".", pd.NA), errors="coerce")
    return result

## Apply the SpliceAI filter

A pure-splicing row:

- contains at least one consequence beginning with `splice_`;
- does not also contain any severe consequence term.

Only pure-splicing rows require a maximum SpliceAI delta score of at least 0.5.

Severe consequences remain regardless of whether SpliceAI is low or unavailable.

In [ ]:
def is_pure_splicing(consequence):
    terms = {term.strip() for term in str(consequence).split("&")}
    return any(term.startswith("splice_") for term in terms) and not bool(terms & severe_terms)

In [ ]:
def filter_spliceai(df):
    result = df.copy()

    result["SpliceAI_max_DS"] = result[spliceai_cols].max(axis=1, skipna=True)
    result["passes_spliceAI"] = result["SpliceAI_max_DS"].ge(spliceai_threshold)
    result["pure_splicing"] = result["Variant.Consequence"].apply(is_pure_splicing)

    removed = result[result["pure_splicing"] & ~result["passes_spliceAI"]].copy()
    filtered = result[(~result["pure_splicing"]) | result["passes_spliceAI"]].copy()

    return result, filtered, removed

## NMD boundary filtering

This section applies the nonsense-mediated decay (NMD) boundary filter after gnomAD, consequence, and SpliceAI filtering.

The goal is to keep candidate loss-of-function variants that occur far enough upstream of the transcript 3′ end to plausibly trigger NMD. Variants in the last exon, variants within the final 50 bp of the penultimate exon, and variants downstream of that boundary toward the transcript 3′ end are not kept as NMD-supporting candidates.

Boundary annotations are matched by both gene and transcript ID. Transcript version numbers are removed before matching so that IDs such as `NM_000059.4` and `NM_000059` can match.

For each variant, the script:

1. keeps only variants in the 16 target breast-cancer genes;
2. merges the variant table with the penultimate-exon boundary file using gene + transcript;
3. defines the final 50 bp window of the penultimate exon in a strand-aware way:
   - `+` strand: final 50 bp is near `ExonEnd`;
   - `-` strand: final 50 bp is near `ExonStart`;
4. classifies each variant relative to this boundary as:
   - `before_boundary`;
   - `within_last50bp_window`;
   - `after_boundary_toward_3prime`;
   - `last_exon`;
   - `missing_boundary_gene_or_transcript_match`;
   - `unable_to_classify`;
5. keeps only variants classified as `before_boundary`.

The full NMD-annotated table is saved for review, and the final filtered table is saved separately as the gnomAD + SpliceAI + NMD-filtered output.

In [ ]:
def clean_transcript_id(x):
    s = str(x).strip()
    return pd.NA if s in {"", ".", "nan", "NaN", "None", "<NA>"} else s.split(".")[0]


#splitting the exon/intron notation, ex. 3|4 into 3, 4
def parse_pair(series):
    parsed = series.astype("string").str.strip().str.replace("/", "|", regex=False).str.extract(r"^(\d+)\|(\d+)")
    return pd.to_numeric(parsed[0], errors="coerce"), pd.to_numeric(parsed[1], errors="coerce")


def filter_nmd(spliceai_filtered):
    boundary = pd.read_csv(nmd_boundary_csv, low_memory=False)
    boundary = boundary.loc[:, ~boundary.columns.duplicated()].copy()

    variants = spliceai_filtered.loc[:, ~spliceai_filtered.columns.duplicated()].copy()
    variants["Gene"] = variants["Gene"].astype("string").str.strip()
    boundary["Gene"] = boundary["Gene"].astype("string").str.strip()

    variants["variant_transcript_clean"] = variants["Feature.Accession"].map(clean_transcript_id)
    boundary["boundary_transcript_clean"] = boundary["TranscriptID"].map(clean_transcript_id)
    boundary = boundary.drop_duplicates(["Gene", "boundary_transcript_clean"], keep="first")

    nmd = variants.merge(
        boundary,
        how="left",
        left_on=["Gene", "variant_transcript_clean"],
        right_on=["Gene", "boundary_transcript_clean"],
        suffixes=("", "_boundary"),
        indicator="boundary_merge_status",
    )

    matched = nmd["boundary_merge_status"].eq("both")
    missing_matches = (
        nmd.loc[~matched, ["Gene", "Feature.Accession", "variant_transcript_clean"]]
        .drop_duplicates()
        .sort_values(["Gene", "variant_transcript_clean"], na_position="last")
    )

    start_col = "Start" if "Start" in nmd.columns else "variant_start"
    end_col = "End" if "End" in nmd.columns else "variant_end"

    nmd["variant_start_pos"] = pd.to_numeric(nmd[start_col], errors="coerce")
    nmd["variant_end_pos"] = pd.to_numeric(nmd[end_col], errors="coerce")
    nmd["variant_exon_num"], nmd["variant_total_exons"] = parse_pair(nmd["EXON"])
    nmd["variant_intron_num"], nmd["variant_total_introns"] = parse_pair(nmd["INTRON"])

    for col in ["Strandness", "ExonStart", "ExonEnd"]:
        nmd[f"{col}_num"] = pd.to_numeric(nmd[col], errors="coerce")

    plus, minus = nmd["Strandness_num"].eq(1), nmd["Strandness_num"].eq(-1)

    nmd["penult_last50_start"], nmd["penult_last50_end"] = np.nan, np.nan
    nmd.loc[plus, "penult_last50_start"] = nmd.loc[plus, "ExonEnd_num"] - 49
    nmd.loc[plus, "penult_last50_end"] = nmd.loc[plus, "ExonEnd_num"]
    nmd.loc[minus, "penult_last50_start"] = nmd.loc[minus, "ExonStart_num"]
    nmd.loc[minus, "penult_last50_end"] = nmd.loc[minus, "ExonStart_num"] + 49

    valid = matched & (plus | minus) & nmd[["penult_last50_start", "penult_last50_end", "variant_start_pos", "variant_end_pos"]].notna().all(axis=1)
    is_last_exon = nmd["variant_exon_num"].notna() & nmd["variant_total_exons"].notna() & nmd["variant_exon_num"].eq(nmd["variant_total_exons"])
    overlaps_last50 = valid & nmd["variant_start_pos"].le(nmd["penult_last50_end"]) & nmd["variant_end_pos"].ge(nmd["penult_last50_start"])

    before_boundary = pd.Series(False, index=nmd.index)
    before_boundary.loc[valid & plus] = nmd.loc[valid & plus, "variant_end_pos"] < nmd.loc[valid & plus, "penult_last50_start"]
    before_boundary.loc[valid & minus] = nmd.loc[valid & minus, "variant_start_pos"] > nmd.loc[valid & minus, "penult_last50_end"]

    after_boundary = pd.Series(False, index=nmd.index)
    after_boundary.loc[valid & plus] = nmd.loc[valid & plus, "variant_start_pos"] > nmd.loc[valid & plus, "penult_last50_end"]
    after_boundary.loc[valid & minus] = nmd.loc[valid & minus, "variant_end_pos"] < nmd.loc[valid & minus, "penult_last50_start"]

    nmd["is_last_exon"] = is_last_exon
    nmd["overlaps_last50_window"] = overlaps_last50
    nmd["before_last50_boundary"] = before_boundary
    nmd["after_last50_boundary_toward_3prime"] = after_boundary

    nmd["nmd_boundary_side"] = "unable_to_classify"
    nmd.loc[~matched, "nmd_boundary_side"] = "missing_boundary_gene_or_transcript_match"
    nmd.loc[matched & is_last_exon, "nmd_boundary_side"] = "last_exon"
    nmd.loc[matched & ~is_last_exon & overlaps_last50, "nmd_boundary_side"] = "within_last50bp_window"
    nmd.loc[matched & ~is_last_exon & ~overlaps_last50 & after_boundary, "nmd_boundary_side"] = "after_boundary_toward_3prime"
    nmd.loc[matched & ~is_last_exon & ~overlaps_last50 & ~after_boundary & before_boundary, "nmd_boundary_side"] = "before_boundary"

    final = nmd[nmd["nmd_boundary_side"].eq("before_boundary")].copy()
    removed = nmd[~nmd["nmd_boundary_side"].eq("before_boundary")].copy()

    return nmd, final, removed, missing_matches

# Filtering for 16 target genes

In [ ]:
def keep_target_genes(df):
    return df[df["Gene"].isin(TARGET_GENES)].copy()

In [ ]:
def add_suffix(filename, suffix):
    path = Path(filename)
    return f"{path.stem}_{suffix}{path.suffix}"

# Filtering Summary

In [ ]:
def make_filtering_summary(run, results):
    summary = pd.DataFrame([
        {"step": "After gnomAD filtering", "rows": len(results["gnomad"])},
        {"step": "After consequence filtering", "rows": len(results["consequence"])},
        {"step": "Pure-splicing rows before SpliceAI filtering", "rows": int(results["spliceai_annotated"]["pure_splicing"].sum())},
        {"step": "Pure-splicing rows passing SpliceAI", "rows": int((results["spliceai_annotated"]["pure_splicing"] & results["spliceai_annotated"]["passes_spliceAI"]).sum())},
        {"step": "Pure-splicing rows removed by SpliceAI", "rows": len(results["spliceai_removed"])},
        {"step": "After gnomAD + SpliceAI filtering", "rows": len(results["spliceai_filtered"])},
        {"step": "Rows removed by NMD filter", "rows": len(results["nmd_removed"])},
        {"step": "Final rows after gnomAD + SpliceAI + NMD", "rows": len(results["final"])},
    ])

    sanity = pd.DataFrame([{
        "run": run,
        "gnomad_filtered_rows": len(results["gnomad"]),
        "consequence_filtered_rows": len(results["consequence"]),
        "spliceai_filtered_rows": len(results["spliceai_filtered"]),
        "final_rows": len(results["final"]),
        "pure_splicing_removed": len(results["spliceai_removed"]),
        "nmd_removed": len(results["nmd_removed"]),
        "spliceai_final_plus_removed_equals_consequence_rows": len(results["spliceai_filtered"]) + len(results["spliceai_removed"]) == len(results["consequence"]),
        "nmd_final_plus_removed_equals_spliceai_rows": len(results["final"]) + len(results["nmd_removed"]) == len(results["spliceai_filtered"]),
    }])

    return summary, sanity

## Process one run

Each run is loaded and filtered independently.

The function retains the row counts needed for the filtering summary but does not add summary fields to the variant tables.

In [ ]:
def process_run(run):
    annovar_file, original_file = get_run_files(run)
    annovar = pd.read_csv(annovar_file, low_memory=False)

    gnomad_filtered, exome_cols, genome_cols = filter_gnomad(annovar)
    consequence_filtered = filter_consequences(gnomad_filtered)
    with_spliceai = clean_spliceai_scores(consequence_filtered)
    spliceai_annotated, spliceai_filtered, spliceai_removed = filter_spliceai(with_spliceai)
    nmd_annotated, final, nmd_removed, nmd_missing_matches = filter_nmd(spliceai_filtered)

    return {
        "gnomad": gnomad_filtered,
        "consequence": consequence_filtered,
        "spliceai_annotated": spliceai_annotated,
        "spliceai_filtered": spliceai_filtered,
        "spliceai_removed": spliceai_removed,
        "nmd_annotated": nmd_annotated,
        "nmd_removed": nmd_removed,
        "nmd_missing_matches": nmd_missing_matches,
        "final": final,
    }

## Save run outputs

The variant files preserve all existing annotation columns plus the calculated SpliceAI fields.

Filtering summaries and sanity checks are saved separately so that QC values do not become variant-level columns.

In [ ]:
def save_run_outputs(run, results):
    run_output_dir = output_dir
    target_output_dir = output_dir / "16_target_genes"
    target_output_dir.mkdir(parents=True, exist_ok=True)

    files = {
        "gnomad": f"{output_prefix}_{run}_gnomad_filtered.csv",
        "consequence": f"{output_prefix}_{run}_gnomad_consequence_filtered.csv",
        "spliceai_filtered": f"{output_prefix}_{run}_gnomad_consequence_spliceai_filtered.csv",
        "final": f"{output_prefix}_{run}_gnomad_consequence_spliceai_NMD_filtered.csv",
        "spliceai_removed": f"{output_prefix}_{run}_spliceai_removed_variants.csv",
        "nmd_annotated": f"{output_prefix}_{run}_NMD_boundary_annotated_all_rows.csv",
        "nmd_removed": f"{output_prefix}_{run}_NMD_removed_variants.csv",
        "nmd_missing_matches": f"{output_prefix}_{run}_missing_NMD_boundary_transcript_matches.csv",
    }

    temporary_cols = ["match_chr", "match_start", "match_ref", "match_alt", "_spliceai_merge"]

    results_16 = {
        key: value[value["Gene"].isin(TARGET_GENES)].copy()
        if isinstance(value, pd.DataFrame) and "Gene" in value.columns
        else value
        for key, value in results.items()
    }

    for key, filename in files.items():
        df = results[key].drop(columns=temporary_cols, errors="ignore")
        df.to_csv(run_output_dir / filename, index=False)

        target_filename = add_suffix(filename, "16_target_genes")
        results_16[key].drop(columns=temporary_cols, errors="ignore").to_csv(target_output_dir / target_filename, index=False)

    summary_all, sanity_all = make_filtering_summary(run, results)
    summary_16, sanity_16 = make_filtering_summary(run, results_16)

    summary_file = output_dir / f"{output_prefix}_{run}_filtering_summary.xlsx"
    sanity_file = output_dir / f"{output_prefix}_{run}_filtering_sanity_check.csv"
    target_summary_file = target_output_dir / f"{output_prefix}_{run}_filtering_summary_16_target_genes.xlsx"
    target_sanity_file = target_output_dir / f"{output_prefix}_{run}_filtering_sanity_check_16_target_genes.csv"

    sanity_all.to_csv(sanity_file, index=False)
    sanity_16.to_csv(target_sanity_file, index=False)

    with pd.ExcelWriter(summary_file) as writer:
        summary_all.to_excel(writer, sheet_name="filtering_counts", index=False)
        sanity_all.to_excel(writer, sheet_name="sanity_checks", index=False)

    with pd.ExcelWriter(target_summary_file) as writer:
        summary_16.to_excel(writer, sheet_name="filtering_counts", index=False)
        sanity_16.to_excel(writer, sheet_name="sanity_checks", index=False)

    print(f"{run}:")
    print(f"  Final rows: {len(results['final']):,}")
    print(f"  Final 16-gene rows: {len(results_16['final']):,}")
    print(f"  Saved full outputs to: {run_output_dir}")
    print(f"  Saved 16-gene outputs to: {target_output_dir}")

    return sanity_all, sanity_16

## Run the filtering workflow

Process and save each selected run independently. A combined sanity-check table is also displayed after all runs finish.

In [ ]:
all_sanity_checks = []
all_sanity_checks_16 = []

for run in runs:
    results = process_run(run)
    sanity_all, sanity_16 = save_run_outputs(run, results)

    all_sanity_checks.append(sanity_all)
    all_sanity_checks_16.append(sanity_16)

combined_sanity_checks = pd.concat(all_sanity_checks, ignore_index=True)
combined_sanity_checks_16 = pd.concat(all_sanity_checks_16, ignore_index=True)

combined_sanity_checks